# Credit Card Fraud Detection — Phase 1 Data Audit

**Audit only:** inspect the source data, repeated rows, and possible train/test overlap. This notebook does not change the CSV, delete rows, or train a model. The original baseline notebook's executable source code is unchanged, but its outputs and metadata have local modifications. Its original committed version remains available in Git history.

## 1. Load tools and locate the data

Pandas reads the CSV as a table (a DataFrame). NumPy, Matplotlib, and Seaborn support analysis and charts. `train_test_split` is used later only to examine duplicate overlap, not to train a model.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "creditcard.csv"
FIGURES_DIR = PROJECT_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Could not find {DATA_PATH}. Open the project folder as the notebook working directory.")
df = pd.read_csv(DATA_PATH)
sns.set_theme(style="whitegrid")
print(f"Loaded {len(df):,} transactions and {df.shape[1]} columns.")

## 2. Inspect the table structure

The shape tells us how many transactions (rows) and fields (columns) there are. The first rows confirm that the CSV loaded correctly. `Class` is the target: 0 means legitimate and 1 means fraud.

In [ ]:
print("Shape:", df.shape)
display(df.head())
print("Column data types:")
display(df.dtypes.rename("dtype").to_frame())

## 3. Check missing values and repeated rows

Missing values are blank or unknown entries. An exact duplicate matches another full row, including its label. We count and study duplicates, but do not remove them: repeated records may affect evaluation, and their meaning should be investigated first.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print("Total missing cells:", int(missing.sum()))
print("Columns with missing values:")
display(missing[missing > 0].to_frame("missing_count"))
repeated_rows = df.duplicated(keep=False)
extra_copies = int(df.duplicated().sum())
exact_groups = df.groupby(list(df.columns), dropna=False, sort=False).size().rename("copies")
repeated_groups = exact_groups[exact_groups > 1]
print(f"Rows inside exact-duplicate groups: {int(repeated_rows.sum()):,}")
print(f"Extra copies beyond the first: {extra_copies:,}")
print(f"Distinct repeated row patterns: {len(repeated_groups):,}")
print(f"Largest copies of one exact row: {int(repeated_groups.max()):,}")
dup_table = repeated_groups.rename_axis(list(df.columns)).reset_index()
dup_table["extra_copy"] = dup_table["copies"] - 1
print("Extra copies by Class:")
display(dup_table.groupby("Class")["extra_copy"].sum().rename("extra_copies"))
feature_cols = [c for c in df.columns if c != "Class"]
conflicts = df.groupby(feature_cols, dropna=False)["Class"].nunique().gt(1).sum()
print("Feature patterns with conflicting labels:", int(conflicts))

## 4. Review statistical summaries

`describe()` reports counts, means, spread, quartiles, and ranges. `V1`–`V28` are anonymized transformed features, so their numbers do not have direct plain-language interpretations here.

In [ ]:
display(df.describe().T.round(3))

## 5. Visualize class imbalance

The left panel uses a logarithmic count axis so the rare fraud class is visible beside the much larger legitimate class. The right panel shows each class as a percentage of all rows.

In [ ]:
counts = df["Class"].value_counts().reindex([0, 1])
percentages = df["Class"].value_counts(normalize=True).reindex([0, 1]) * 100
labels = ["Legitimate (0)", "Fraud (1)"]
colors = ["#4C78A8", "#E45756"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, values, title, ylabel in [(axes[0], counts.values, "Transaction counts (log scale)", "Rows"), (axes[1], percentages.values, "Share of all transactions", "Percent")]:
    bars = ax.bar(labels, values, color=colors)
    ax.set(title=title, ylabel=ylabel)
    if ax is axes[0]: ax.set_yscale("log")
    for bar, value in zip(bars, values):
        label = f"{value:,}" if ax is axes[0] else f"{value:.3f}%"
        ax.annotate(label, (bar.get_x()+bar.get_width()/2, value), xytext=(0, 5), textcoords="offset points", ha="center")
fig.suptitle("Severe class imbalance", y=1.03)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "class_distribution_v2.png", dpi=180, bbox_inches="tight")
plt.show()

## 6. Explore transaction amount

Amounts are right-skewed: a small number are far larger than a typical transaction. `log1p` means `log(1 + Amount)`. It is used only to make the chart easier to read; the source data is unchanged.

In [ ]:
amount_plot = df.assign(log_amount=np.log1p(df["Amount"]), class_label=df["Class"].map({0:"Legitimate", 1:"Fraud"}))
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=amount_plot, x="log_amount", hue="class_label", stat="density", common_norm=False, bins=60, element="step", fill=False, palette={"Legitimate":"#4C78A8", "Fraud":"#E45756"}, ax=ax)
ax.set(title="Transaction amount distributions by class", xlabel="log(1 + Amount) — visual scale only", ylabel="Within-class density")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "amount_distribution_v2.png", dpi=180, bbox_inches="tight")
plt.show()

## 7. Explore transaction time

`Time` is seconds from the start of this dataset. We convert it to hours for the chart. The data covers roughly two days, so this is a short-window description, not evidence of long-term patterns.

In [ ]:
time_plot = df.assign(hours=df["Time"] / 3600, class_label=df["Class"].map({0:"Legitimate", 1:"Fraud"}))
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=time_plot, x="hours", hue="class_label", stat="density", common_norm=False, bins=48, element="step", fill=False, palette={"Legitimate":"#4C78A8", "Fraud":"#E45756"}, ax=ax)
ax.set(title="Transaction time distribution by class", xlabel="Hours from dataset start", ylabel="Within-class density")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "transaction_time_distribution_v2.png", dpi=180, bbox_inches="tight")
plt.show()

## 8. Check duplicate overlap across train/test splits

We simulate the old unstratified 70/30 split and a stratified 70/30 split using a fixed random seed. This is a split audit only; no model is trained. A row fingerprint lets us count identical complete rows appearing on both sides. Stratification preserves class proportions, but it does not prevent duplicate overlap.

In [ ]:
row_fingerprints = pd.util.hash_pandas_object(df, index=False)
def duplicate_split_report(stratified):
    indexes = np.arange(len(df))
    train_i, test_i = train_test_split(indexes, test_size=0.30, random_state=42, stratify=df["Class"] if stratified else None)
    train_fingerprints = set(row_fingerprints.iloc[train_i].tolist())
    test_fingerprints = row_fingerprints.iloc[test_i]
    matched = test_fingerprints.isin(train_fingerprints)
    fraud_test = df["Class"].iloc[test_i].to_numpy() == 1
    return {"split":"Stratified 70/30" if stratified else "Original unstratified 70/30",
            "repeated_patterns_in_both":len(set(test_fingerprints.loc[matched].tolist())),
            "test_rows_with_train_match":int(matched.sum()),
            "fraud_test_rows_with_train_match":int((matched.to_numpy() & fraud_test).sum()),
            "fraud_in_test":int(fraud_test.sum())}
overlap = pd.DataFrame([duplicate_split_report(False), duplicate_split_report(True)])
display(overlap.set_index("split"))
print("These are duplicate-overlap counts, not model metrics.")

## 9. Visualize duplicate overlap

The chart compares repeated patterns and affected test rows. If an identical example appears in training and test, evaluation may partly reward recognition of a copy. That does not prove the row is invalid, so we do not remove it in this phase.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, column, title, ylabel in [(axes[0], "repeated_patterns_in_both", "Repeated row patterns in both splits", "Patterns"), (axes[1], "test_rows_with_train_match", "Test rows matching a training row", "Test rows")]:
    bars = ax.bar(overlap["split"], overlap[column], color=["#F2CF5B", "#72B7B2"])
    ax.set(title=title, ylabel=ylabel)
    ax.tick_params(axis="x", rotation=10)
    for bar, value in zip(bars, overlap[column]): ax.annotate(f"{value:,}", (bar.get_x()+bar.get_width()/2, value), xytext=(0, 5), textcoords="offset points", ha="center")
fig.suptitle("Exact duplicate overlap can cross a random split", y=1.03)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "duplicate_split_overlap_v2.png", dpi=180, bbox_inches="tight")
plt.show()

## 10. Phase 1 findings and handoff

The current CSV contains **284,807 rows**, **31 columns**, **no missing cells**, and **492 fraud cases (about 0.173%)**. It has **1,081 extra exact copies** across **773 repeated row patterns**; the largest pattern occurs 18 times. In the fixed 70/30 random split, identical rows appear on both sides. Stratification preserves the fraud proportion but does not eliminate that overlap.

No rows were removed and no model was trained. Phase 2 should choose and justify a duplicate policy, use stratified evaluation, and put learned preprocessing inside model pipelines.